# 01 - Limpieza Star Wars BI

Este notebook prepara los datos para Power BI. No hace storytelling ni conclusiones de negocio: solo carga datos raw, normaliza nombres/claves, limpia la encuesta y exporta tablas base a `data/processed`.

Salida principal:
- tablas largas de encuesta (`survey_*`)
- tablas limpias del universo (`universe_*_clean`)
- tabla de activos (`universe_assets`)
- tabla comercial de peliculas (`films_business_clean`)


## 1. Configuracion y carga de datos


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import unicodedata


def find_project_root(start_path=None):
    """Busca la raiz del proyecto para evitar rutas absolutas dependientes del usuario."""
    start_path = Path(start_path or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate
    return start_path


BASE_DIR = find_project_root()
os.chdir(BASE_DIR)

print("Ahora Python esta en:")
print(Path.cwd())

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [ ]:
RAW_DIR = Path("data/raw") #definimos la carpeta base donde se encuentran los datos en bruto, 


# Creamos la ruta completa al CSV principal de la encuesta de Star Wars.
STARWARS_PATH = RAW_DIR / "StarWars" / "StarWars.csv" 


# Creamos la ruta a la carpeta donde están los CSV extra descargados de Kaggle.
KAGGLE_CSV_DIR = RAW_DIR / "starwars_kaggle" / "archive (1)" / "csv"

print("Ruta StarWars:", STARWARS_PATH) #mostramos por pantalla la ruta del archivo principal
print("Existe StarWars:", STARWARS_PATH.exists()) #devuelve True si el archivo existe, False si no

print("Ruta Kaggle CSV:", KAGGLE_CSV_DIR) 
print("Existe carpeta Kaggle CSV:", KAGGLE_CSV_DIR.exists()) #comprobamos si existe o no


#comprobamos si existe una archivo en especifico

print("Existe characters.csv:", (KAGGLE_CSV_DIR / "characters.csv").exists()) 

#pasamos a leer con pandas los archivos csv de las database que hemos elegido,
#el resultado se guarda en un dataframe para cada uno de los archivos
df_starwars_raw= pd.read_csv(STARWARS_PATH)

df_characters = pd.read_csv(KAGGLE_CSV_DIR / "characters.csv")
df_films = pd.read_csv(KAGGLE_CSV_DIR / "films.csv")
df_planets = pd.read_csv(KAGGLE_CSV_DIR / "planets.csv")
df_species = pd.read_csv(KAGGLE_CSV_DIR / "species.csv")
df_starships = pd.read_csv(KAGGLE_CSV_DIR / "starships.csv")
df_vehicles = pd.read_csv(KAGGLE_CSV_DIR / "vehicles.csv")
df_quotes = pd.read_csv(KAGGLE_CSV_DIR / "quotes.csv")
df_weapons = pd.read_csv(KAGGLE_CSV_DIR / "weapons.csv")
df_droids = pd.read_csv(KAGGLE_CSV_DIR / "droids.csv")


print("\nDatasets cargados correctamente:")

#mostramos los tamaños de los datasets cargados, el número de filas y columnas de cada uno

print("Characters:", df_characters.shape)
print("Films:", df_films.shape)
print("Planets:", df_planets.shape)
print("Species:", df_species.shape)
print("Starships:", df_starships.shape)
print("Vehicles:", df_vehicles.shape)
print("Quotes:", df_quotes.shape)
print("Weapons:", df_weapons.shape)
print("Droids:", df_droids.shape)
#mostramos el tamaño del dataset principal de la encuesta de star wars 
#y las primeras filas para comprobar que se ha cargado correctamente
print("Dimensiones originales:", df_starwars_raw.shape)
display(df_starwars_raw.head())


## 2. Funciones de limpieza y claves canonicas


In [ ]:
# FUNCIONES GENERALES DE LIMPIEZA


def normalize_column_names(df):
    """Normaliza nombres de columnas a snake_case basico."""
    df = df.copy()
    normalized_columns = []

    for column in df.columns:
        column_name = str(column).strip().lower()
        column_name = unicodedata.normalize("NFKD", column_name).encode("ascii", "ignore").decode("ascii")
        column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
        column_name = re.sub(r"_+", "_", column_name).strip("_")
        normalized_columns.append(column_name)

    df.columns = normalized_columns
    return df


def clean_text(value):
    """Limpia un valor de texto individual y convierte falsos nulos en NaN."""
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    if value == "" or value.lower() in ["nan", "none", "null"]:
        return np.nan
    return value


def clean_text_columns(df):
    """Limpia todas las columnas de texto de un DataFrame."""
    df = df.copy()
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        df[column] = df[column].apply(clean_text)

    return df


def apply_column_mapping(df, column_mapping):
    """Renombra columnas usando un diccionario de equivalencias."""
    df = df.copy()
    existing_mapping = {
        source_column: target_column
        for source_column, target_column in column_mapping.items()
        if source_column in df.columns
    }
    return df.rename(columns=existing_mapping)


def add_missing_columns(df, required_columns):
    """Anade columnas necesarias que no existan en el dataset."""
    df = df.copy()

    for column in required_columns:
        if column not in df.columns:
            df[column] = np.nan

    return df


def normalize_text_key(value):
    """Convierte cualquier etiqueta de texto en una clave estable tipo snake_case."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value if value else np.nan


CHARACTER_KEY_ALIASES = {
    "princess_leia_organa": "leia_organa",
    "c3po": "c_3po",
    "c_3_po": "c_3po",
    "r2d2": "r2_d2",
    "r2_d_2": "r2_d2",
    "obiwan_kenobi": "obi_wan_kenobi",
    "sheev_palpatine": "emperor_palpatine",
    "palpatine": "emperor_palpatine",
    "queen_amidala": "padme_amidala",
}

CHARACTER_DISPLAY_NAMES = {
    "han_solo": "Han Solo",
    "luke_skywalker": "Luke Skywalker",
    "leia_organa": "Leia Organa",
    "anakin_skywalker": "Anakin Skywalker",
    "obi_wan_kenobi": "Obi-Wan Kenobi",
    "emperor_palpatine": "Emperor Palpatine",
    "darth_vader": "Darth Vader",
    "lando_calrissian": "Lando Calrissian",
    "boba_fett": "Boba Fett",
    "c_3po": "C-3PO",
    "r2_d2": "R2-D2",
    "jar_jar_binks": "Jar Jar Binks",
    "padme_amidala": "Padme Amidala",
    "yoda": "Yoda",
}

FILM_KEY_ALIASES = {
    "episode_i_the_phantom_menace": "the_phantom_menace",
    "episode_ii_attack_of_the_clones": "attack_of_the_clones",
    "episode_iii_revenge_of_the_sith": "revenge_of_the_sith",
    "episode_iv_a_new_hope": "a_new_hope",
    "episode_v_the_empire_strikes_back": "the_empire_strikes_back",
    "episode_vi_return_of_the_jedi": "return_of_the_jedi",
    "episode_vii_the_force_awakens": "the_force_awakens",
    "episode_viii_the_last_jedi": "the_last_jedi",
    "episode_ix_the_rise_of_skywalker": "the_rise_of_skywalker",
    "star_wars_ep_i_the_phantom_menace": "the_phantom_menace",
    "star_wars_ep_ii_attack_of_the_clones": "attack_of_the_clones",
    "star_wars_ep_iii_revenge_of_the_sith": "revenge_of_the_sith",
    "star_wars_ep_iv_a_new_hope": "a_new_hope",
    "star_wars_ep_v_the_empire_strikes_back": "the_empire_strikes_back",
    "star_wars_ep_vi_return_of_the_jedi": "return_of_the_jedi",
    "star_wars_ep_vii_the_force_awakens": "the_force_awakens",
    "star_wars_ep_viii_the_last_jedi": "the_last_jedi",
    "star_wars_the_rise_of_skywalker": "the_rise_of_skywalker",
    "rogue_one_a_star_wars_story": "rogue_one",
    "solo_a_star_wars_story": "solo",
    "star_wars_the_mandalorian_and_grogu": "the_mandalorian_and_grogu",
}

FILM_DISPLAY_NAMES = {
    "the_phantom_menace": "Episode I: The Phantom Menace",
    "attack_of_the_clones": "Episode II: Attack of the Clones",
    "revenge_of_the_sith": "Episode III: Revenge of the Sith",
    "a_new_hope": "Episode IV: A New Hope",
    "the_empire_strikes_back": "Episode V: The Empire Strikes Back",
    "return_of_the_jedi": "Episode VI: Return of the Jedi",
    "the_force_awakens": "Episode VII: The Force Awakens",
    "the_last_jedi": "Episode VIII: The Last Jedi",
    "the_rise_of_skywalker": "Episode IX: The Rise of Skywalker",
    "rogue_one": "Rogue One: A Star Wars Story",
    "solo": "Solo: A Star Wars Story",
    "the_mandalorian_and_grogu": "The Mandalorian and Grogu",
}


def normalize_character_key(value):
    key = normalize_text_key(value)
    if pd.isna(key):
        return np.nan
    return CHARACTER_KEY_ALIASES.get(key, key)


def canonical_character_name(value):
    key = normalize_character_key(value)
    if pd.isna(key):
        return np.nan
    return CHARACTER_DISPLAY_NAMES.get(key, str(value).strip())


def normalize_film_key(value):
    key = normalize_text_key(value)
    if pd.isna(key):
        return np.nan
    return FILM_KEY_ALIASES.get(key, key)


def canonical_film_title(value):
    key = normalize_film_key(value)
    if pd.isna(key):
        return np.nan
    return FILM_DISPLAY_NAMES.get(key, str(value).strip())


def split_list_values(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


def normalize_list_text(value, canonical_func):
    items = split_list_values(value)
    if not items:
        return np.nan
    return ", ".join(canonical_func(item) for item in items)


def normalize_list_keys(value, key_func):
    items = split_list_values(value)
    if not items:
        return np.nan
    return ", ".join(key_func(item) for item in items)


def add_key_column(df, source_col, key_col, key_func=normalize_text_key):
    df = df.copy()
    if source_col in df.columns:
        df[key_col] = df[source_col].apply(key_func)
    return df


## 3. Limpieza de encuesta


In [ ]:
# INSPECCIÓN ESPECIAL DEL DATASET DE ENCUESTA STAR WARS


# Guardamos una copia del dataset original de la encuesta.
# Así no perdemos nunca la estructura tal como venía en el CSV.

df_survey = df_starwars_raw.copy()

# Guardamos la primera fila porque parece contener información de opciones/subpreguntas.
# No parece una respuesta real de una persona.

metadata_row = df_starwars_raw.iloc[0]

# Creamos una versión de trabajo de la encuesta quitando la fila 0.
# Esta será la tabla con respuestas reales.

df_survey = df_starwars_raw.drop(index=0).reset_index(drop=True)
print("Dimensiones originales:", df_starwars_raw.shape)
print("Dimensiones encuesta limpia de filas:", df_survey.shape)

display(df_survey.head())


In [ ]:
# REPORTE DE NULOS DE LA ENCUESTA STAR WARS


def missing_report(df):
    """
    Crea un informe de valores nulos por columna.
    Muestra:
    - número de nulos
    - porcentaje de nulos
    Solo muestra columnas que tienen al menos un nulo.
    """
    return (
        pd.DataFrame({
            "nulos": df.isna().sum(),
            "porcentaje": (df.isna().mean() * 100).round(2),
        })
        .query("nulos > 0")
        .sort_values("porcentaje", ascending=False)
    )


# Aplicamos el reporte solo a la encuesta limpia, sin la fila 0 de metadata.
missing_survey = missing_report(df_survey)

display(missing_survey)


In [ ]:
# 6. NORMALIZAR NOMBRES DE COLUMNAS DE LA ENCUESTA


df_survey = normalize_column_names(df_survey)

print("Columnas normalizadas:")
for i, col in enumerate(df_survey.columns):
    print(i, "->", col)


In [ ]:
# 7. LIMPIEZA BÁSICA DE TEXTOS
'''quita espacios al principio/final
convierte textos vacíos en NaN
convierte "nan", "none", "null" en NaN real
'''

df_survey = clean_text_columns(df_survey)

display(df_survey.head())


In [ ]:
# RENOMBRADO MANUAL DE COLUMNAS DE LA ENCUESTA


survey_column_mapping = {
    # Identificador
    df_survey.columns[0]: "respondent_id",

    # Preguntas generales
    df_survey.columns[1]: "has_seen_any_star_wars_film",
    df_survey.columns[2]: "is_star_wars_fan",

    # Películas vistas
    df_survey.columns[3]: "seen_episode_i_the_phantom_menace",
    df_survey.columns[4]: "seen_episode_ii_attack_of_the_clones",
    df_survey.columns[5]: "seen_episode_iii_revenge_of_the_sith",
    df_survey.columns[6]: "seen_episode_iv_a_new_hope",
    df_survey.columns[7]: "seen_episode_v_the_empire_strikes_back",
    df_survey.columns[8]: "seen_episode_vi_return_of_the_jedi",

    # Ranking de películas
    df_survey.columns[9]: "rank_episode_i_the_phantom_menace",
    df_survey.columns[10]: "rank_episode_ii_attack_of_the_clones",
    df_survey.columns[11]: "rank_episode_iii_revenge_of_the_sith",
    df_survey.columns[12]: "rank_episode_iv_a_new_hope",
    df_survey.columns[13]: "rank_episode_v_the_empire_strikes_back",
    df_survey.columns[14]: "rank_episode_vi_return_of_the_jedi",

    # Opinión sobre personajes
    df_survey.columns[15]: "opinion_han_solo",
    df_survey.columns[16]: "opinion_luke_skywalker",
    df_survey.columns[17]: "opinion_princess_leia_organa",
    df_survey.columns[18]: "opinion_anakin_skywalker",
    df_survey.columns[19]: "opinion_obi_wan_kenobi",
    df_survey.columns[20]: "opinion_emperor_palpatine",
    df_survey.columns[21]: "opinion_darth_vader",
    df_survey.columns[22]: "opinion_lando_calrissian",
    df_survey.columns[23]: "opinion_boba_fett",
    df_survey.columns[24]: "opinion_c_3po",
    df_survey.columns[25]: "opinion_r2_d2",
    df_survey.columns[26]: "opinion_jar_jar_binks",
    df_survey.columns[27]: "opinion_padme_amidala",
    df_survey.columns[28]: "opinion_yoda",

    # Preguntas generales finales
    df_survey.columns[29]: "which_character_shot_first",
    df_survey.columns[30]: "is_familiar_with_expanded_universe",
    df_survey.columns[31]: "is_expanded_universe_fan",
    df_survey.columns[32]: "is_star_trek_fan",

    # Datos demográficos
    df_survey.columns[33]: "gender",
    df_survey.columns[34]: "age",
    df_survey.columns[35]: "household_income",
    df_survey.columns[36]: "education",
    df_survey.columns[37]: "location_census_region",
    df_survey.columns[38]: "extra_column_38",
    df_survey.columns[39]: "extra_column_39",
}

df_survey = df_survey.rename(columns=survey_column_mapping)

print("Columnas renombradas correctamente:")
for i, col in enumerate(df_survey.columns):
    print(i, "->", col)


In [ ]:
# GRUPOS DE COLUMNAS CON NOMBRES DEFINITIVOS


general_columns = [
    "has_seen_any_star_wars_film",
    "is_star_wars_fan",
    "which_character_shot_first",
    "is_familiar_with_expanded_universe",
    "is_expanded_universe_fan",
    "is_star_trek_fan",
]

seen_movies_columns = [
    "seen_episode_i_the_phantom_menace",
    "seen_episode_ii_attack_of_the_clones",
    "seen_episode_iii_revenge_of_the_sith",
    "seen_episode_iv_a_new_hope",
    "seen_episode_v_the_empire_strikes_back",
    "seen_episode_vi_return_of_the_jedi",
]

ranking_columns = [
    "rank_episode_i_the_phantom_menace",
    "rank_episode_ii_attack_of_the_clones",
    "rank_episode_iii_revenge_of_the_sith",
    "rank_episode_iv_a_new_hope",
    "rank_episode_v_the_empire_strikes_back",
    "rank_episode_vi_return_of_the_jedi",
]

character_opinion_columns = [
    "opinion_han_solo",
    "opinion_luke_skywalker",
    "opinion_princess_leia_organa",
    "opinion_anakin_skywalker",
    "opinion_obi_wan_kenobi",
    "opinion_emperor_palpatine",
    "opinion_darth_vader",
    "opinion_lando_calrissian",
    "opinion_boba_fett",
    "opinion_c_3po",
    "opinion_r2_d2",
    "opinion_jar_jar_binks",
    "opinion_padme_amidala",
    "opinion_yoda",
]

demographic_columns = [
    "gender",
    "age",
    "household_income",
    "education",
    "location_census_region",
]

survey_column_groups = {
    "general": general_columns,
    "seen_movies": seen_movies_columns,
    "ranking": ranking_columns,
    "character_opinion": character_opinion_columns,
    "demographic": demographic_columns,
}

for group_name, columns in survey_column_groups.items():
    print("\n" + "=" * 70)
    print(group_name.upper())
    print("=" * 70)
    for col in columns:
        print(col)


In [ ]:
# LIMPIEZA FINAL DE LA ENCUESTA

# Partimos de df_survey, que ya tiene la fila 0 eliminada y las columnas renombradas.
df_survey_clean = df_survey.copy()

# 1. Reconstruccion de demografia.
# Los ingresos venian partidos por comas: $50,000 - $99,999 se reparte en varias columnas.
VALID_GENDERS = {"male", "female"}
VALID_AGES = {"18-29", "30-44", "45-60", "> 60"}
VALID_EDUCATION = {
    "less than high school degree",
    "high school degree",
    "some college or associate degree",
    "bachelor degree",
    "graduate degree",
}
VALID_REGIONS = {
    "new england", "middle atlantic", "east north central", "west north central",
    "south atlantic", "east south central", "west south central", "mountain", "pacific",
}


def _clean_token(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    return np.nan if value in ["", "nan", "none", "null"] else value


def _first_valid(tokens, valid_values):
    for token in tokens:
        if token in valid_values:
            return token
    return np.nan


def _rebuild_income(tokens):
    if "$0 - $24" in tokens and "999" in tokens:
        return "$0 - $24,999"
    if "$25" in tokens and "000 - $49" in tokens and "999" in tokens:
        return "$25,000 - $49,999"
    if "$50" in tokens and "000 - $99" in tokens and "999" in tokens:
        return "$50,000 - $99,999"
    if "$100" in tokens and "000 - $149" in tokens and "999" in tokens:
        return "$100,000 - $149,999"
    if "$150" in tokens and "000+" in tokens:
        return "$150,000+"
    return np.nan


def _rebuild_demographics(row):
    tail_cols = [
        "is_expanded_universe_fan", "is_star_trek_fan", "gender", "age",
        "household_income", "education", "location_census_region",
        "extra_column_38", "extra_column_39",
    ]
    tokens = [_clean_token(row[col]) for col in tail_cols if col in row.index]
    tokens = [token for token in tokens if isinstance(token, str)]
    return pd.Series({
        "gender": _first_valid(tokens, VALID_GENDERS),
        "age": _first_valid(tokens, VALID_AGES),
        "household_income": _rebuild_income(tokens),
        "education": _first_valid(tokens, VALID_EDUCATION),
        "location_census_region": _first_valid(tokens, VALID_REGIONS),
    })

fixed_demographics = df_survey_clean.apply(_rebuild_demographics, axis=1)
for col in fixed_demographics.columns:
    df_survey_clean[col] = fixed_demographics[col]

df_survey_clean = df_survey_clean.drop(
    columns=[col for col in ["extra_column_38", "extra_column_39"] if col in df_survey_clean.columns]
)

# 2. Peliculas vistas: texto = vista, NaN = no seleccionada.
for col in seen_movies_columns:
    df_survey_clean[col] = df_survey_clean[col].notna().astype(int)

df_survey_clean["total_movies_seen"] = df_survey_clean[seen_movies_columns].sum(axis=1)

# 3. Rankings de peliculas a numerico.
for col in ranking_columns:
    df_survey_clean[col] = pd.to_numeric(df_survey_clean[col], errors="coerce")

df_survey_clean["has_complete_movie_ranking"] = (
    df_survey_clean[ranking_columns].notna().sum(axis=1).eq(6).astype(int)
)

# 4. Opiniones de personajes a puntuacion numerica.
opinion_score_map = {
    "very favorably": 2,
    "somewhat favorably": 1,
    "neither favorably nor unfavorably (neutral)": 0,
    "somewhat unfavorably": -1,
    "very unfavorably": -2,
    "unfamiliar (n/a)": np.nan,
}
opinion_score_columns = []
for col in character_opinion_columns:
    score_col = col.replace("opinion_", "opinion_score_")
    df_survey_clean[score_col] = df_survey_clean[col].map(opinion_score_map)
    opinion_score_columns.append(score_col)

df_survey_clean["average_character_opinion_score"] = df_survey_clean[opinion_score_columns].mean(axis=1)

# 5. Variables auxiliares para analisis ejecutivo.
def yes_no_to_binary(value):
    if value == "yes":
        return 1
    if value == "no":
        return 0
    return np.nan

for col in [
    "has_seen_any_star_wars_film", "is_star_wars_fan",
    "is_familiar_with_expanded_universe", "is_expanded_universe_fan", "is_star_trek_fan",
]:
    df_survey_clean[col + "_binary"] = df_survey_clean[col].apply(yes_no_to_binary)


def movie_consumption_segment(total_movies):
    if total_movies == 0:
        return "no_movies_seen"
    if total_movies <= 2:
        return "low_consumption"
    if total_movies <= 4:
        return "medium_consumption"
    return "high_consumption"


df_survey_clean["movie_consumption_segment"] = df_survey_clean["total_movies_seen"].apply(movie_consumption_segment)
df_survey_clean["fan_segment"] = np.where(
    df_survey_clean["is_star_wars_fan"] == "yes", "fan",
    np.where(df_survey_clean["is_star_wars_fan"] == "no", "not_fan", "unknown")
)
df_survey_clean["has_demographic_info"] = df_survey_clean[[
    "gender", "age", "household_income", "education", "location_census_region"
]].notna().any(axis=1).astype(int)

print("Dimensiones encuesta limpia:", df_survey_clean.shape)
print("Duplicados completos:", df_survey_clean.duplicated().sum())
print("IDs duplicados:", df_survey_clean["respondent_id"].duplicated().sum())

print("\nDistribucion de peliculas vistas:")
display(df_survey_clean["total_movies_seen"].value_counts().sort_index())

print("\nCategorias demograficas corregidas:")
for col in ["gender", "age", "household_income", "education", "location_census_region"]:
    print("\n" + col)
    display(df_survey_clean[col].value_counts(dropna=False))

print("\nNulos principales tras limpieza:")
display(missing_report(df_survey_clean).head(20))


## 4. Tablas largas de encuesta para Power BI


In [ ]:
# TABLAS LARGAS PARA POWER BI

PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

movie_metadata = {
    "seen_episode_i_the_phantom_menace": ("Episode I: The Phantom Menace", "the_phantom_menace", 1),
    "seen_episode_ii_attack_of_the_clones": ("Episode II: Attack of the Clones", "attack_of_the_clones", 2),
    "seen_episode_iii_revenge_of_the_sith": ("Episode III: Revenge of the Sith", "revenge_of_the_sith", 3),
    "seen_episode_iv_a_new_hope": ("Episode IV: A New Hope", "a_new_hope", 4),
    "seen_episode_v_the_empire_strikes_back": ("Episode V: The Empire Strikes Back", "the_empire_strikes_back", 5),
    "seen_episode_vi_return_of_the_jedi": ("Episode VI: Return of the Jedi", "return_of_the_jedi", 6),
}
rank_metadata = {col: movie_metadata[col.replace("rank_", "seen_")] for col in ranking_columns}

character_metadata = {
    "opinion_han_solo": ("han_solo", "Han Solo"),
    "opinion_luke_skywalker": ("luke_skywalker", "Luke Skywalker"),
    "opinion_princess_leia_organa": ("leia_organa", "Leia Organa"),
    "opinion_anakin_skywalker": ("anakin_skywalker", "Anakin Skywalker"),
    "opinion_obi_wan_kenobi": ("obi_wan_kenobi", "Obi-Wan Kenobi"),
    "opinion_emperor_palpatine": ("emperor_palpatine", "Emperor Palpatine"),
    "opinion_darth_vader": ("darth_vader", "Darth Vader"),
    "opinion_lando_calrissian": ("lando_calrissian", "Lando Calrissian"),
    "opinion_boba_fett": ("boba_fett", "Boba Fett"),
    "opinion_c_3po": ("c_3po", "C-3PO"),
    "opinion_r2_d2": ("r2_d2", "R2-D2"),
    "opinion_jar_jar_binks": ("jar_jar_binks", "Jar Jar Binks"),
    "opinion_padme_amidala": ("padme_amidala", "Padme Amidala"),
    "opinion_yoda": ("yoda", "Yoda"),
}

survey_respondents = df_survey_clean[[
    "respondent_id", "has_seen_any_star_wars_film", "has_seen_any_star_wars_film_binary",
    "is_star_wars_fan", "is_star_wars_fan_binary", "fan_segment",
    "total_movies_seen", "movie_consumption_segment", "has_complete_movie_ranking",
    "which_character_shot_first", "is_familiar_with_expanded_universe",
    "is_familiar_with_expanded_universe_binary", "is_expanded_universe_fan",
    "is_expanded_universe_fan_binary", "is_star_trek_fan", "is_star_trek_fan_binary",
    "gender", "age", "household_income", "education", "location_census_region",
    "has_demographic_info", "average_character_opinion_score",
]].copy()

survey_movies_seen = df_survey_clean[["respondent_id"] + seen_movies_columns].melt(
    id_vars="respondent_id", var_name="movie_code", value_name="has_seen_movie"
)
survey_movies_seen["movie_title"] = survey_movies_seen["movie_code"].map({k: v[0] for k, v in movie_metadata.items()})
survey_movies_seen["film_key"] = survey_movies_seen["movie_code"].map({k: v[1] for k, v in movie_metadata.items()})
survey_movies_seen["episode_order"] = survey_movies_seen["movie_code"].map({k: v[2] for k, v in movie_metadata.items()})

survey_movie_rankings = df_survey_clean[["respondent_id"] + ranking_columns].melt(
    id_vars="respondent_id", var_name="movie_code", value_name="movie_rank"
)
survey_movie_rankings["movie_title"] = survey_movie_rankings["movie_code"].map({k: v[0] for k, v in rank_metadata.items()})
survey_movie_rankings["film_key"] = survey_movie_rankings["movie_code"].map({k: v[1] for k, v in rank_metadata.items()})
survey_movie_rankings["episode_order"] = survey_movie_rankings["movie_code"].map({k: v[2] for k, v in rank_metadata.items()})
survey_movie_rankings["has_ranking"] = survey_movie_rankings["movie_rank"].notna().astype(int)

survey_character_opinions = df_survey_clean[["respondent_id"] + character_opinion_columns].melt(
    id_vars="respondent_id", var_name="character_code", value_name="opinion_label"
)
survey_character_opinions["character_key"] = survey_character_opinions["character_code"].map({k: v[0] for k, v in character_metadata.items()})
survey_character_opinions["character_name"] = survey_character_opinions["character_code"].map({k: v[1] for k, v in character_metadata.items()})
survey_character_opinions["opinion_score"] = survey_character_opinions["opinion_label"].map(opinion_score_map)
survey_character_opinions["has_character_opinion"] = survey_character_opinions["opinion_label"].notna().astype(int)
survey_character_opinions["is_unfamiliar"] = survey_character_opinions["opinion_label"].eq("unfamiliar (n/a)").astype(int)

print("survey_respondents:", survey_respondents.shape)
print("survey_movies_seen:", survey_movies_seen.shape)
print("survey_movie_rankings:", survey_movie_rankings.shape)
print("survey_character_opinions:", survey_character_opinions.shape)

display(survey_respondents.head())
display(survey_movies_seen.head())
display(survey_movie_rankings.head())
display(survey_character_opinions.head())

# Exportacion provisional para Power BI.
df_survey_clean.to_csv(PROCESSED_DIR / "survey_clean_wide.csv", index=False)
survey_respondents.to_csv(PROCESSED_DIR / "survey_respondents.csv", index=False)
survey_movies_seen.to_csv(PROCESSED_DIR / "survey_movies_seen.csv", index=False)
survey_movie_rankings.to_csv(PROCESSED_DIR / "survey_movie_rankings.csv", index=False)
survey_character_opinions.to_csv(PROCESSED_DIR / "survey_character_opinions.csv", index=False)

print("CSV de encuesta exportados en:", PROCESSED_DIR)


## 5. Limpieza de datasets del universo


In [ ]:
# LIMPIEZA GENERAL DE DATASETS DEL UNIVERSO

PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

UNKNOWN_VALUES = {
    "", "unknown", "n/a", "na", "none", "null", "nan", "not available", "not applicable"
}


def clean_universe_text_columns(df):
    """Limpia textos y convierte falsos nulos en NaN."""
    df = df.copy()
    text_columns = df.select_dtypes(include=["object", "string", "str"]).columns

    for col in text_columns:
        df[col] = df[col].apply(clean_text)
        df[col] = df[col].apply(
            lambda value: np.nan
            if isinstance(value, str) and value.strip().lower() in UNKNOWN_VALUES
            else value
        )

    return df


def to_numeric_columns(df, columns):
    """Convierte columnas a numerico si existen."""
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def count_list_items(value):
    """Cuenta elementos separados por coma en columnas tipo lista."""
    if pd.isna(value):
        return 0
    value = str(value).strip()
    if value == "":
        return 0
    if value.lower() == "all episodes":
        return 11
    return len([item for item in value.split(",") if item.strip()])


def add_list_count(df, source_col, target_col):
    """Crea una columna con numero de elementos en una lista textual."""
    df = df.copy()
    if source_col in df.columns:
        df[target_col] = df[source_col].apply(count_list_items)
    return df


def clean_year_column(value):
    """Extrae valores numericos de anos tipo '0 BBY' o '34'."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return np.nan
    return float(match.group())


universe_raw_datasets = {
    "characters": df_characters,
    "films": df_films,
    "planets": df_planets,
    "species": df_species,
    "starships": df_starships,
    "vehicles": df_vehicles,
    "quotes": df_quotes,
    "weapons": df_weapons,
    "droids": df_droids,
}

universe_clean_datasets = {}

for dataset_name, dataset in universe_raw_datasets.items():
    df = normalize_column_names(dataset)
    df = clean_universe_text_columns(df)
    df = df.drop_duplicates().reset_index(drop=True)
    universe_clean_datasets[dataset_name] = df

# Conversiones especificas por dataset.

universe_clean_datasets["characters"] = to_numeric_columns(
    universe_clean_datasets["characters"],
    ["id", "height", "weight"],
)
for col in ["year_born", "year_died"]:
    if col in universe_clean_datasets["characters"].columns:
        universe_clean_datasets["characters"][col] = universe_clean_datasets["characters"][col].apply(clean_year_column)

universe_clean_datasets["films"]["release_date"] = pd.to_datetime(
    universe_clean_datasets["films"]["release_date"], errors="coerce"
)
universe_clean_datasets["films"]["release_year"] = universe_clean_datasets["films"]["release_date"].dt.year

universe_clean_datasets["planets"] = to_numeric_columns(
    universe_clean_datasets["planets"],
    ["id", "diameter", "rotation_period", "orbital_period", "population", "surface_water"],
)
universe_clean_datasets["planets"] = add_list_count(universe_clean_datasets["planets"], "residents", "resident_count")
universe_clean_datasets["planets"] = add_list_count(universe_clean_datasets["planets"], "films", "film_count")

universe_clean_datasets["species"] = to_numeric_columns(
    universe_clean_datasets["species"],
    ["id", "average_height", "average_lifespan"],
)

universe_clean_datasets["starships"] = to_numeric_columns(
    universe_clean_datasets["starships"],
    [
        "id", "cost_in_credits", "length", "max_atmosphering_speed", "crew",
        "passengers", "cargo_capacity", "hyperdrive_rating", "mglt", "MGLT",
    ],
)
universe_clean_datasets["starships"] = add_list_count(universe_clean_datasets["starships"], "pilots", "pilot_count")
universe_clean_datasets["starships"] = add_list_count(universe_clean_datasets["starships"], "films", "film_count")

universe_clean_datasets["vehicles"] = to_numeric_columns(
    universe_clean_datasets["vehicles"],
    ["id", "cost_in_credits", "length", "max_atmosphering_speed", "crew", "passengers", "cargo_capacity"],
)
universe_clean_datasets["vehicles"] = add_list_count(universe_clean_datasets["vehicles"], "pilots", "pilot_count")
universe_clean_datasets["vehicles"] = add_list_count(universe_clean_datasets["vehicles"], "films", "film_count")

universe_clean_datasets["quotes"] = to_numeric_columns(universe_clean_datasets["quotes"], ["id"])

universe_clean_datasets["weapons"] = to_numeric_columns(
    universe_clean_datasets["weapons"],
    ["id", "cost_in_credits", "length"],
)
universe_clean_datasets["weapons"] = add_list_count(universe_clean_datasets["weapons"], "films", "film_count")

universe_clean_datasets["droids"] = to_numeric_columns(
    universe_clean_datasets["droids"],
    ["id", "height", "mass"],
)
universe_clean_datasets["droids"] = add_list_count(universe_clean_datasets["droids"], "films", "film_count")

# Claves normalizadas para relaciones y etiquetas canonicas.
universe_clean_datasets["characters"]["character_key"] = universe_clean_datasets["characters"]["name"].apply(normalize_character_key)
universe_clean_datasets["characters"]["name"] = universe_clean_datasets["characters"]["name"].apply(canonical_character_name)

universe_clean_datasets["films"]["film_key"] = universe_clean_datasets["films"]["title"].apply(normalize_film_key)
universe_clean_datasets["films"]["title"] = universe_clean_datasets["films"]["title"].apply(canonical_film_title)

universe_clean_datasets["planets"] = add_key_column(universe_clean_datasets["planets"], "name", "planet_key")
universe_clean_datasets["planets"]["resident_keys"] = universe_clean_datasets["planets"]["residents"].apply(lambda value: normalize_list_keys(value, normalize_character_key))
universe_clean_datasets["planets"]["residents"] = universe_clean_datasets["planets"]["residents"].apply(lambda value: normalize_list_text(value, canonical_character_name))
universe_clean_datasets["planets"]["film_keys"] = universe_clean_datasets["planets"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

universe_clean_datasets["species"] = add_key_column(universe_clean_datasets["species"], "name", "species_key")

universe_clean_datasets["starships"] = add_key_column(universe_clean_datasets["starships"], "name", "starship_key")
universe_clean_datasets["starships"]["pilot_keys"] = universe_clean_datasets["starships"]["pilots"].apply(lambda value: normalize_list_keys(value, normalize_character_key))
universe_clean_datasets["starships"]["pilots"] = universe_clean_datasets["starships"]["pilots"].apply(lambda value: normalize_list_text(value, canonical_character_name))
universe_clean_datasets["starships"]["film_keys"] = universe_clean_datasets["starships"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

universe_clean_datasets["vehicles"] = add_key_column(universe_clean_datasets["vehicles"], "name", "vehicle_key")
universe_clean_datasets["vehicles"]["pilot_keys"] = universe_clean_datasets["vehicles"]["pilots"].apply(lambda value: normalize_list_keys(value, normalize_character_key))
universe_clean_datasets["vehicles"]["pilots"] = universe_clean_datasets["vehicles"]["pilots"].apply(lambda value: normalize_list_text(value, canonical_character_name))
universe_clean_datasets["vehicles"]["film_keys"] = universe_clean_datasets["vehicles"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

universe_clean_datasets["quotes"]["character_key"] = universe_clean_datasets["quotes"]["character_name"].apply(normalize_character_key)
universe_clean_datasets["quotes"]["character_name"] = universe_clean_datasets["quotes"]["character_name"].apply(canonical_character_name)
universe_clean_datasets["quotes"]["source_film_key"] = universe_clean_datasets["quotes"]["source"].apply(normalize_film_key)

universe_clean_datasets["weapons"] = add_key_column(universe_clean_datasets["weapons"], "name", "weapon_key")
universe_clean_datasets["weapons"]["film_keys"] = universe_clean_datasets["weapons"]["films"].apply(
    lambda value: "all_episodes" if isinstance(value, str) and value.strip().lower() == "all episodes" else normalize_list_keys(value, normalize_film_key)
)

universe_clean_datasets["droids"]["droid_key"] = universe_clean_datasets["droids"]["name"].apply(normalize_character_key)
universe_clean_datasets["droids"]["name"] = universe_clean_datasets["droids"]["name"].apply(canonical_character_name)
universe_clean_datasets["droids"]["film_keys"] = universe_clean_datasets["droids"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

# Recuperamos variables comodas para seguir trabajando en el notebook.
df_characters_clean = universe_clean_datasets["characters"]
df_films_clean = universe_clean_datasets["films"]
df_planets_clean = universe_clean_datasets["planets"]
df_species_clean = universe_clean_datasets["species"]
df_starships_clean = universe_clean_datasets["starships"]
df_vehicles_clean = universe_clean_datasets["vehicles"]
df_quotes_clean = universe_clean_datasets["quotes"]
df_weapons_clean = universe_clean_datasets["weapons"]
df_droids_clean = universe_clean_datasets["droids"]

print("Datasets del universo limpiados:")
for name, df in universe_clean_datasets.items():
    print(f"{name}: {df.shape[0]} filas x {df.shape[1]} columnas")


In [ ]:
# REPORTE DE CALIDAD DE LOS DATASETS DEL UNIVERSO

universe_quality_summary = []
universe_missing_reports = {}

for name, df in universe_clean_datasets.items():
    duplicate_rows = df.duplicated().sum()
    duplicate_names = df["name"].duplicated().sum() if "name" in df.columns else np.nan
    total_cells = df.shape[0] * df.shape[1]
    total_missing = int(df.isna().sum().sum())
    missing_pct = round((total_missing / total_cells) * 100, 2) if total_cells else 0

    universe_quality_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": duplicate_rows,
        "duplicate_names": duplicate_names,
        "total_missing": total_missing,
        "missing_pct": missing_pct,
    })

    universe_missing_reports[name] = missing_report(df)

universe_quality_summary = pd.DataFrame(universe_quality_summary)

display(universe_quality_summary)

print("Columnas con mas nulos por dataset:")
for name, report in universe_missing_reports.items():
    print("\n" + "=" * 70)
    print(name.upper())
    display(report.head(10))


In [ ]:
# TABLA RESUMEN DE ACTIVOS DEL UNIVERSO PARA POWER BI

# Esta tabla unifica activos de distinto tipo para rankings y KPIs ejecutivos.
# No sustituye a las tablas detalladas; sirve para comparar presencia general.

asset_tables = []

asset_sources = {
    "character": (df_characters_clean, "name", "character_key"),
    "planet": (df_planets_clean, "name", "planet_key"),
    "species": (df_species_clean, "name", "species_key"),
    "starship": (df_starships_clean, "name", "starship_key"),
    "vehicle": (df_vehicles_clean, "name", "vehicle_key"),
    "weapon": (df_weapons_clean, "name", "weapon_key"),
    "droid": (df_droids_clean, "name", "droid_key"),
}

for asset_type, (df, name_col, key_col) in asset_sources.items():
    temp = pd.DataFrame({
        "asset_key": df[key_col] if key_col in df.columns else df[name_col].apply(normalize_text_key),
        "asset_name": df[name_col] if name_col in df.columns else pd.Series([np.nan] * len(df)),
    })
    temp["asset_type"] = asset_type
    temp["source_dataset"] = asset_type + "s"
    temp["film_count"] = df["film_count"] if "film_count" in df.columns else np.nan
    temp["known_numeric_fields"] = df.select_dtypes(include="number").notna().sum(axis=1).values
    temp["missing_fields"] = df.isna().sum(axis=1).values
    temp["total_fields"] = df.shape[1]
    temp["data_completeness_pct"] = ((temp["total_fields"] - temp["missing_fields"]) / temp["total_fields"] * 100).round(2)
    asset_tables.append(temp)

universe_assets = pd.concat(asset_tables, ignore_index=True)

# Indicador simple de presencia interna. Mas adelante se podra combinar con afinidad de audiencia.
universe_assets["internal_presence_score"] = (
    universe_assets["film_count"].fillna(0) + universe_assets["known_numeric_fields"].fillna(0)
)

display(universe_assets.head())
print("Activos por tipo:")
display(universe_assets["asset_type"].value_counts())


In [ ]:
# EXPORTACION DE DATASETS DEL UNIVERSO LIMPIOS

for name, df in universe_clean_datasets.items():
    df.to_csv(PROCESSED_DIR / f"universe_{name}_clean.csv", index=False)

universe_quality_summary.to_csv(PROCESSED_DIR / "universe_quality_summary.csv", index=False)
universe_assets.to_csv(PROCESSED_DIR / "universe_assets.csv", index=False)

print("CSV del universo exportados en:", PROCESSED_DIR)
print("Archivos creados:")
for file in sorted(PROCESSED_DIR.glob("universe_*.csv")):
    print("-", file.name)


## 6. Tabla comercial de peliculas


In [ ]:
# DATOS COMERCIALES DE PELICULAS

from io import StringIO

film_business_csv = """film_key,film_title,the_numbers_title,release_date,era,film_type,budget_usd,domestic_box_office_usd,worldwide_box_office_usd,data_status
"a_new_hope","A New Hope","Star Wars Ep. IV: A New Hope",1977-05-25,original_trilogy,saga_episode,11000000,460998007,775398007,final
"the_empire_strikes_back","The Empire Strikes Back","Star Wars Ep. V: The Empire Strikes Back",1980-05-20,original_trilogy,saga_episode,23000000,291738960,549001086,final
"return_of_the_jedi","Return of the Jedi","Star Wars Ep. VI: Return of the Jedi",1983-05-25,original_trilogy,saga_episode,32500000,316465003,482365284,final
"the_phantom_menace","The Phantom Menace","Star Wars Ep. I: The Phantom Menace",1999-05-19,prequel_trilogy,saga_episode,115000000,487574671,1046513456,final
"attack_of_the_clones","Attack of the Clones","Star Wars Ep. II: Attack of the Clones",2002-05-16,prequel_trilogy,saga_episode,115000000,310676740,656695615,final
"revenge_of_the_sith","Revenge of the Sith","Star Wars Ep. III: Revenge of the Sith",2005-05-18,prequel_trilogy,saga_episode,115000000,414378291,902891983,final
"the_force_awakens","The Force Awakens","Star Wars Ep. VII: The Force Awakens",2015-12-16,sequel_trilogy,saga_episode,533200000,936662225,2056046835,final
"rogue_one","Rogue One","Rogue One: A Star Wars Story",2016-12-14,disney_anthology,spin_off,280200000,533539991,1055083596,final
"the_last_jedi","The Last Jedi","Star Wars Ep. VIII: The Last Jedi",2017-12-13,sequel_trilogy,saga_episode,262000000,620181382,1322581071,final
"solo","Solo","Solo: A Star Wars Story",2018-05-23,disney_anthology,spin_off,330400000,213767512,393151347,final
"the_rise_of_skywalker","The Rise of Skywalker","Star Wars: The Rise of Skywalker",2019-12-18,sequel_trilogy,saga_episode,275000000,515202542,1069951814,final
"the_mandalorian_and_grogu","The Mandalorian and Grogu","Star Wars: The Mandalorian and Grogu",2026-05-20,new_theatrical_release,streaming_series_continuation,165000000,158382333,296046513,partial_current_release
"""

films_business_clean = pd.read_csv(StringIO(film_business_csv), parse_dates=["release_date"])
films_business_clean["release_year"] = films_business_clean["release_date"].dt.year
films_business_clean["profit_estimated_usd"] = films_business_clean["worldwide_box_office_usd"] - films_business_clean["budget_usd"]
films_business_clean["roi"] = (films_business_clean["profit_estimated_usd"] / films_business_clean["budget_usd"]).round(4)
films_business_clean["source_name"] = "The Numbers"
films_business_clean["source_url"] = "https://www.the-numbers.com/movies/franchise/Star-Wars"

films_business_clean = films_business_clean[[
    "film_key", "film_title", "the_numbers_title", "release_date", "release_year", "era", "film_type",
    "budget_usd", "domestic_box_office_usd", "worldwide_box_office_usd", "profit_estimated_usd", "roi",
    "data_status", "source_name", "source_url",
]]

eda_film_business_summary = films_business_clean.sort_values(
    ["worldwide_box_office_usd", "roi"], ascending=False
).reset_index(drop=True)

films_business_clean.to_csv(PROCESSED_DIR / "films_business_clean.csv", index=False)

print("Tabla comercial de peliculas:", films_business_clean.shape)
display(eda_film_business_summary)


## Resultado

Al terminar este notebook, Power BI puede leer las tablas limpias desde `data/processed`. El siguiente paso analitico esta en `02_eda_storytelling_star_wars.ipynb`.
